# COMPLEX PCA ON fMRI TIME-SERIES

The following code will apply a *Complex Principal Component Analysis* framework to decompose fMRI time-series into a series of
spatiotemporal patterns of propagation. From these patterns, we identify the *N* dominant ones (here, *N* = 4), and further characterize their properties. We specifically focus on their propagation duration as well as regional power. 

This code represents a minimal framework of the code used for the publication: **Fotiadis P, Jang H, Dai R, Li D, Cofré R, Timmermann C, Nutt DJ, Carhart-Harris RL, Mashour GA, Hudetz AG, Huang Z. "Reorganization of Human Brain Waves Across Diverse States of Consciousness" Submitted (2026)**, and has been adapted to run for any given fMRI matrix of size: time-points x brain regions. It assumes that the fMRI time-series have already been preprocessed and denoised. fMRI parameters specific to the acquisition (such as repetition time) can be specified under "Inputs and Parameters" below. 

Other relevant publications that might be useful to the interested reader -- which have also helped shape the code below -- are:
1. Bolt, T., Nomi, J.S., Bzdok, D. et al. A parsimonious description of global functional brain organization in three spatiotemporal patterns. Nat Neurosci 25, 1093–1103 (2022) -> github code: https://github.com/tsb46/BOLD_WAVES/blob/master/figures.ipynb
2. Byeon, K., Park, H., Park, S. et al. Developmental variations in recurrent spatiotemporal brain propagations from childhood to adulthood. Nat Commun 17, 1012 (2026) -> github code: https://github.com/HumanBrainED/Neurodev-CPCA/blob/main/scripts/cpca.py

# LIBRARIES

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import hilbert, periodogram
from scipy.stats import median_abs_deviation
from scipy.io import loadmat

# INPUTS AND PARAMETERS

In [5]:
# --------------------------------------------
# -> Adapt the following parameters as needed
# --------------------------------------------

# Define the path to the fMRI BOLD matrix to be analyzed
file_path = "/Users/your_user_name/data" # parent directory where all input files are stored
input_file = Path(file_path) / "fmri_timeseries.npy"  # time-points × regions

# fMRI-specific parameters
TR = 2.0                              # repetition time (seconds)
standardize = True                    # whether you would like to z-score each brain region across time
n_components = 4                      # number of complex principal components (i.e., dominant patterns of propagation) to focus on
n_bins = 32                           # phase bins spanning [0, 2π)

# Outputs
output_path = Path(file_path) / "cpca_results" # output directory
save_outputs = True


# LOAD AND PREPARE THE MATRIX

In [ ]:
# Load the fMRI matrix
BOLD = np.load(input_file)

if BOLD.ndim != 2 or np.iscomplexobj(BOLD):
    raise ValueError("The input must be a real matrix with shape time-points × regions")

BOLD = np.asarray(BOLD, dtype=float)

# Extract number of time-points and brain regions
n_timepoints, num_ROIs = BOLD.shape

# Perform some QC checks, just in case
if n_timepoints < 3 or num_ROIs < 1:
    raise ValueError("At least three time-points and one region are needed.")
if not np.isfinite(TR) or TR <= 0:
    raise ValueError("TR must be a positive finite number in seconds.")
if not isinstance(n_components, int) or n_components < 1:
    raise ValueError("n_components must be a positive integer.")

# Keep only regions with finite values and non-zero variation across time
valid_regions = np.isfinite(BOLD).all(axis=0) & (BOLD.std(axis=0) > 0)

if not valid_regions.any():
    raise ValueError("No finite, non-constant regions remain.")
    
X = BOLD[:, valid_regions].copy()

# Subtract each region’s temporal mean from its time-series (giving each column a mean of zero)
X = X - X.mean(axis=0)

# If enabled, divide each region’s centered time-series by its sample standard deviation. Combined with the previous step, 
# this step effectively z-scores each region across time.
if standardize:
    X /= X.std(axis=0, ddof=1)

print(f"{n_timepoints} time-points × {num_ROIs} regions; TR = {TR:g} s")
print(f"Included {valid_regions.sum()} regions")


# COMPLEX PCA

In [6]:
# This function performs Singular Value Decomposition (SVD) on an input matrix. 
# When applying SVD in real matrices, you might deal with the sign ambiguity issue (https://www.educative.io/blog/sign-ambiguity-in-singular-value-decomposition)
# The analogous issue when applying SVD to complex matrices is the phase ambiguity issue. 
# In a complex SVD, each pair of corresponding columns in U and rows in Vt can be multiplied by a complex unit with magnitude 1 (i.e., 
# exp(i*theta), where theta is any real angle) without changing the product U*S*Vt; thus, instead of flipping a sign as you would do 
# in the real matrix case, you can just apply a unitary phase factor.

def svd_with_phase_correction(A, tolerance=1e-9):

    # Input:
    # (Hilbert-transformed) complex-valued fMRI matrix A (time-points × valid brain regions)

    # Outputs: 
    # U_corrected: temporal singular vectors (time-points × K), where K = min(number of time-points, number of valid regions)
    # S: singular values (length K), indicating the strength of each component, ordered from largest to smallest
    # Vt_corrected: complex spatial vectors (K × valid regions), with phases adjusted to preserve the reconstruction
    # Together: A = U_corrected @ np.diag(S) @ Vt_corrected.
    
    # Perform SVD
    U, S, Vt = np.linalg.svd(A, full_matrices=False)

    U_corrected = U.copy()
    Vt_corrected = Vt.copy()
    
    # Apply phase correction to address the potential phase ambiguity issue
    for i in range(U.shape[1]):
        
        # Determine the phase of the first element in the i-th column
        phase_correction = np.exp(-1j * np.angle(U[0, i]))  # this is a unit norm complex number whose angle is opposite the phase of U[0,i]

        # Apply phase correction to both U and Vt
        U_corrected[:, i] *= phase_correction # so when you multiply U[:,i] by the above, all elements in that column get rotated in the complex plane by that opposite angle while setting the angle of the first element to zero. The first element of this corrected column is now real and non-negative. 
        Vt_corrected[i, :] *= phase_correction.conjugate()

    # Verify reconstruction with modified phases
    A_reconstructed = np.dot(U_corrected, np.dot(np.diag(S), Vt_corrected))
    reconstruction_error = np.linalg.norm(A - A_reconstructed)
    
    # Check if the reconstruction error exceeds the specified tolerance
    if reconstruction_error > tolerance:
        raise ValueError(f"Reconstruction error {reconstruction_error} exceeds tolerance {tolerance}!")

    return U_corrected, S, Vt_corrected
    

In [7]:
# This function reconstructs phase-binned spatiotemporal patterns from the complex-valued principal component time-series 
# U and spatial patterns Vt, returning a 3D tensor that captures how spatial activity evolves over the phase cycle of each component.

# This code was adapted from https://github.com/HumanBrainED/Neurodev-CPCA/blob/main/scripts/cpca.py 
# (corresponding manuscript: https://www.nature.com/articles/s41467-025-67754-w)

def reconstruct(u, Vt, n_bins=n_bins):

    # Inputs:
    # u: complex component scores (time-points × components), computed as U * S
    # Vt: corresponding complex spatial vectors (components × brain regions)
    # n_bins: number of bins dividing the component's temporal phase over [0, 2π)
    
    # Output:
    # phase_map: complex phase-binned patterns (components × phase bins × brain regions);
    # phase_map[k, b, r] is the expected fMRI signal at region r when component k is at phase bin b of its complex rotation
        
    n_tps, n_comp = u.shape
    angle_u = np.mod(np.angle(u), 2 * np.pi) # Extract phases from the (complex) time-series
    bins = np.linspace(0, 2 * np.pi, n_bins + 1)

    # Get the bin index for each element in angle_u (this assigns each time-point of each component to one of n_bins phase bins)
    bin_idx = np.digitize(angle_u, bins) - 1  # (n_tps, n_comp)
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)  # prevents out-of-bounds indexing

    # For each component, average the complex scores within each phase bin
    
    # Initialize sum and count arrays
    sum_phase_u = np.zeros((n_bins, n_comp), dtype=np.complex128)
    count_phase_u = np.zeros((n_bins, n_comp), dtype=int)

    # Vectorized summing for each bin and component
    for comp in range(n_comp):
        np.add.at(sum_phase_u[:, comp], bin_idx[:, comp], u[:, comp])
        np.add.at(count_phase_u[:, comp], bin_idx[:, comp], 1)

    # Compute the mean of each bin (-> represents the mean complex coefficient of each component at each phase of its cycle)
    mean_phase_u = np.divide(sum_phase_u, count_phase_u, out=np.full_like(sum_phase_u, np.nan), where=count_phase_u != 0) # size: n_bins x PCs

    # Reconstruct the spatial map per phase bin per component
    phase_map = mean_phase_u[:, :, np.newaxis] * Vt[np.newaxis, :, :]
    phase_map = np.transpose(phase_map, (1, 0, 2)) # (PCs, n_bins, ROIs)
    
    return phase_map
    

In [ ]:
# ---------------------------------------------
# Run the Complex Principal Component Analysis
# ---------------------------------------------

# Compute the Hilbert transform along time
complex_data = hilbert(X, axis=0)

# Apply Singular Value Decomposition
U, S, Vt_kept = svd_with_phase_correction(complex_data)

# Compute the explained variance:
# How much of the total variability in the data is captured by each principal component (or singular vector)
explained_variance_ratio = S**2 / np.sum(S**2)

# Define the number of (complex) printipal components of interest
NC = min(n_components, len(S))

if NC < n_components:
    print(f"Retaining {NC} components (the maximum available).")

# Principal component scores represent how each time-point expresses each component
pc_scores = U[:, :NC] * S[:NC] # shape: (time-points, principal components)

# Restore excluded regions to keep the original region ordering
Vt = np.full((NC, num_ROIs), np.nan + 0j)
Vt[:, valid_regions] = Vt_kept[:NC]

# Loadings represent how strongly each brain region contributes to each component, normalized
loadings = Vt * S[:NC, None] / np.sqrt(n_timepoints - 1) # shape: (regions, components)

# Reconstruct spatiotemporal patterns
patterns = reconstruct(pc_scores, Vt, n_bins)  # shape: (components × bins × regions)

# (Normalized) regional power of each spatiotemporal pattern (i.e., complex principal component)
power = np.abs(Vt)**2
power /= np.nansum(power, axis=1, keepdims=True)

# Store the CPCA outputs
cpca_output = {
    "S": S[:NC], "Vt": Vt, "loadings": loadings, "explained_variance_ratio": explained_variance_ratio, 
    "pc_scores": pc_scores, "patterns": patterns, "power": power
}

# Print out how much variance of the data the first N components capture
print(f"First {NC} components explain {100 * explained_variance_ratio[:NC].sum():.1f}% of the variance.")


# CYCLE DURATION AND SPATIAL PHASE RANGE

In [ ]:
# This function computes the average cycle duration (in seconds), as well as traveling and standing components of each propagation pattern 
# (principal component), after the Complex Principal Component Analysis framework has been run.

# A complementary definition of cycle duration can be found in https://github.com/tsb46/BOLD_WAVES/blob/master/figures.ipynb and its accompanying 
# manuscript: https://www.nature.com/articles/s41593-022-01118-1

def duration_secs(pc_scores, Vt, TR=TR, n_bins=n_bins):

    # Inputs:
    #     pc_scores: complex component time-series, with shape (time-points,)
    #     Vt: complex spatial vector for the same component, with shape (regions,)
    #     TR: repetition time of the fMRI sequence in seconds
    #     n_bins: number of phase bins used in CPCA visualization

    # Outputs:
    #     travel_comp_duration: traveling component duration (seconds)
    #     stand_comp_duration: standing component duration (seconds)
    #     spatial_span_frac: spatial phase span as a fraction of a cycle
    #     cycle_durations: array of individual cycle durations (seconds)
    #     num_cycles: number of complete cycles detected between crossings
    #     avg_cycle_dur: Overall duration (in seconds) of the spatiotemporal pattern. Computed as the median duration across all cycles for each spatiotemporal pattern.

    def _phase_cross_times(phase_unwrapped, TR):
        
        """Time-points (in seconds) when unwrapped phase crosses integer multiples of 2π, via linear interpolation."""
        
        t = np.arange(len(phase_unwrapped)) * TR

        # Normalize the unwrapped phases to cycles
        phi0 = phase_unwrapped[0]
        z = (phase_unwrapped - phi0) / (2 * np.pi)  # cycles since start (units of cycles, not radians)

        # Identify the integer cycle crossings
        k = np.arange(1, np.floor(z.max()) + 1)

        if k.size == 0:
            return np.array([])

        # Find when each crossing happens (interpolation)
        idx = np.array([np.flatnonzero((z[:-1] < level) & (z[1:] >= level))[0] + 1 for level in k], dtype=int)

        z0, z1 = z[idx - 1], z[idx]
        t0, t1 = t[idx - 1], t[idx]

        # If z1 == z0 anywhere (flat), interpolation would blow -> guard it
        denom = z1 - z0
        bad = denom == 0
        if np.any(bad):
            denom = denom.astype(float)
            denom[bad] = np.nan

        # Linearly interpolate between the samples that bracket each crossing
        w = (k - z0) / denom

        # Return the times in seconds when each full cycle finishes
        return t0 + w * (t1 - t0)


    def spatial_phase_range(Vt, k=3.0):
        """
        Estimate the spatial phase span of a complex CPCA component.

        The function:
        1) Removes ROIs with unusually low amplitudes (where phase is unstable),
        2) computes the circular phase span across the remaining ROIs, and
        3) returns the median span across all possible reference choices.
        """

        # Ensure array format
        Vt = np.asarray(Vt)

        # Extract phase and amplitude from the complex spatial vector
        phases = np.angle(Vt)  # phase of each ROI (−π, π]
        amp = np.abs(Vt)       # amplitude (strength) of the component at each ROI

        # Keep only entries with finite values
        valid = np.isfinite(phases) & np.isfinite(amp)
        if valid.sum() < 3:  # need at least 3 regions to estimate a span
            return np.nan

        # Robust center (median) and spread estimate
        med = np.nanmedian(amp[valid])

        robust_std = median_abs_deviation(amp[valid], scale="normal", nan_policy="omit")

        # Keep only ROIs with sufficiently large amplitudes (small amplitudes tend to have unstable / noisy phase as per the literature)
        if robust_std == 0 or not np.isfinite(robust_std):
            keep = valid
        else:
            keep = valid & (amp > (med - k * robust_std))

        if keep.sum() < 3:
            return np.nan

        # Phases that remain after filtering
        phases_k = phases[keep]
        spans = []

        # Evaluate the phase span using every ROI as a reference
        for a in phases_k:

            # Shift phases relative to reference 'a' and wrap back to (-π, π]
            shifted = np.angle(np.exp(1j * (phases_k - a)))

            # Circular span = max phase - min phase after wrapping
            spans.append(np.nanmax(shifted) - np.nanmin(shifted))

        return float(np.nanmedian(spans))


    # Perform some QC checks    
    pc_scores = np.asarray(pc_scores)
    Vt = np.asarray(Vt)

    if pc_scores.ndim != 1:
        raise ValueError("pc_scores must be 1D for a single component (shape (T,))")
    if Vt.ndim != 1:
        raise ValueError("Vt must be 1D for a single component (shape (R,))")
    if n_bins < 1:
        raise ValueError("n_bins must be >= 1")
    if not np.isfinite(TR) or TR <= 0:
        raise ValueError("TR must be a positive finite number")

    # -------------------
    # SPATIAL PHASE RANGE
    # -------------------

    # Phase range as a fraction of a cycle
    spatial_span_frac = spatial_phase_range(Vt) / (2 * np.pi)

    # ---------------
    # CYCLE DURATION
    # ---------------

    # Unwrap temporal phase angles
    comp_phase = np.unwrap(np.angle(pc_scores))

    # Default outputs when no complete cycles can be estimated
    travel_comp_duration = np.nan
    stand_comp_duration = np.nan
    cycle_durations = np.nan
    num_cycles = np.nan
    avg_cycle_dur = np.nan

    if comp_phase.size >= 2 and np.isfinite(comp_phase).all():

        # 1) Sub-TR accurate cycle boundaries via interpolation
        cross_times = _phase_cross_times(comp_phase, TR)

        # Need at least two crossings to have one full-cycle duration
        # (as we avoid assuming that the signal starts at a phase multiple of 2π).
        if cross_times.size >= 2:

            # 2) Cycle durations (seconds) and their median
            cycle_durations = np.diff(cross_times)

            avg_cycle_dur = float(np.nanmedian(cycle_durations))

            # 3) Number of cycles
            num_cycles = int(cycle_durations.size)

            # ------------------------------------------------------------
            # SPLIT CYCLE DURATION INTO TRAVELLING AND STANDING COMPONENTS
            # ------------------------------------------------------------
            
            # 4) Compute the traveling and standing components of the wave
            travel_comp_duration = spatial_span_frac * avg_cycle_dur
            stand_comp_duration = avg_cycle_dur - travel_comp_duration

    return travel_comp_duration, stand_comp_duration, spatial_span_frac, cycle_durations, num_cycles, avg_cycle_dur
    

In [ ]:
# Now that CPCA has been ran, compute each component's duration 

# Use each component's complex time-series and spatial vectors to calculate its duration measures
values = [duration_secs(pc_scores[:, pc], Vt[pc], TR, n_bins) for pc in range(NC)]

# Name the outputs in the same order as they are returned by duration_secs
metric_names = ["travel_comp_duration", "stand_comp_duration", "spatial_span_frac", "cycle_durations", "num_cycles", "avg_cycle_dur"]

# Organize the results into a table, with one row per component
metrics = pd.DataFrame(values, columns=metric_names, index=[f"CPC{pc + 1}" for pc in range(NC)])
metrics.index.name = "component"

# Display the summary measures rounded to three decimals
display(metrics.drop(columns="cycle_durations").round(3))


# VISUALS

In [ ]:
# Plot the cumulative explained variance accumulated as you add complex principal components

fig, ax = plt.subplots()

# Cumulative explained variance
cumulative_variance = np.nancumsum(cpca_output['explained_variance_ratio'][:NC])

# Principal components
components = np.arange(1, NC + 1)

# Bar plot: individual explained variance
ax.bar(components, cpca_output['explained_variance_ratio'][:NC], color='steelblue', label='Individual Explained Variance')
ax.set_xlabel('Complex Principal Component')
ax.set_xticks(range(NC+1))
ax.set_ylim(0, 1)

# Twin axis for cumulative variance
ax2 = ax.twinx()
ax2.plot(components, cumulative_variance, color='orange', marker='o', label='Cumulative Explained Variance')
ax2.axhline(y=0.5, color='red', linestyle='--', label='50% threshold')
ax2.set_ylim(0, 1.05)

ax.set_ylabel('Explained Variance Ratio')
ax2.set_ylabel('Cumulative Explained Variance')

# Combined legend
lines, labels = [], []
for ax in [ax, ax2]:
    line, label = ax.get_legend_handles_labels()
    lines += line
    labels += label
ax.legend(lines, labels, loc='upper right')

fig.tight_layout()

plt.show()

print(f"Cumulative explained variances: {cumulative_variance}")


# SAVE OUTPUTS

In [ ]:
# Save the results, if saving is enabled
if save_outputs:
    
    # Create the output directory, including any missing parent directories
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Save the CPCA outputs, the mask of included regions, and the analysis parameters
    np.savez_compressed(output_path / "cpca_results.npz", **cpca_output, valid_regions=valid_regions, TR=TR, n_bins=n_bins, standardize=standardize)
    
    # Save the summary measures as a CSV table, with one row per component
    metrics.drop(columns="cycle_durations").to_csv(output_path / "cpca_metrics.csv")
    
    print(f"Saved outputs to {output_path.resolve()}")
    